### ATracker Tutorial

This is a compact tutorial demonstrating the full ATracker workflow with the tutorial videos, covering setup, interactive annotation, tracking, processing, manual tracking, and batch measurement.

#### Quick Start — Track a Single Video

For a single video you don't need a project folder, overview file, or config. `track_video` handles everything automatically:

1. Extracts a background image from the video (in memory)
2. Optionally opens GUIs to draw a region of interest and exclusion mask
3. Opens the threshold calibration GUI on random frames
4. Tracks and writes **only** the CSV (and optional `_TR.mp4`) alongside your video — no other files are created

All intermediate data (background, mask, thresholds) are temporary. Use `save_settings` to save them to a single `.atcache` file if you want to re-track the same video later without repeating the setup GUIs.

Use the full `ATracker` workflow (sections 1–11 below) when you have many videos, need batch processing, or want to persist settings across sessions.

In [ ]:
import atracker

# Minimal — opens threshold GUI, tracks, writes CSV and _TR.mp4 alongside the video
atracker.track_video("/path/to/video.mp4")

# With ROI and mask drawing
atracker.track_video(
    "/path/to/video.mp4",
    method="bw",          # "bw" for background subtraction, or a colour name e.g. "red"
    n_objects=1,          # number of objects to track
    draw_roi=True,        # open GUI to draw region of interest (default: full frame)
    draw_mask=True,       # open GUI to draw exclusion mask
    set_threshold=True,   # open threshold calibration GUI
    simple=True,          # True = centroid only; False = full shape (head/tail/orientation)
    show_tracking=False,  # show live tracking window
    create_vid=True,      # write _TR.mp4 tracking overlay
    frame_start=None,     # first frame (default: 1)
    frame_stop=None,      # last frame (default: end of video)
    fps=None,             # override recorded fps
    overwrite=True,
    save_settings="my_video.atcache",  # save bg/mask/thresholds/roi for later re-use
)

# Re-track the same video using saved settings — skips all setup GUIs
atracker.track_video(
    "/path/to/video.mp4",
    load_settings="my_video.atcache",
    set_threshold=False,  # reuse saved threshold
)

# Load or inspect a settings bundle directly
settings = atracker.load_atcache("my_video.atcache")
# settings["img_bg"]    — background image (numpy array)
# settings["img_mask"]  — mask image (numpy array) or None
# settings["threshinfo"]— threshold dict
# settings["roi"]       — ((x1,y1),(x2,y2))

#### 1. Load ATracker and Initialise

In [ ]:
import os
import atracker

home_dir = os.path.expanduser("~")
video_folder = os.path.join(home_dir, "Desktop", "tutorial")

AT = atracker.ATracker(video_folder)

#### 2. Setup Files

Register all videos in the project folder. Provide `fname_vars` and `fname_sep` to extract metadata from filenames automatically. Use `fps` to override the frame rate stored in the file (useful when the recorded fps differs from the intended fps).

In [ ]:
# Extract variables from filenames, e.g. exp_name_date_session_time.mp4
AT.setup_files(fname_vars=("exp", "name", "date", "session", "time"), fname_sep="_")

In [ ]:
# Override fps when the recorded value is wrong
AT.setup_files(fname_vars=("exp", "name", "date", "session", "time"), fname_sep="_", fps=25)

After making changes to the overview Excel file directly, reload it:


In [ ]:
AT.reload()  # re-read overview.xlsx and config from disk

In [ ]:
AT.save()    # write any in-memory overview changes back to disk

#### 3. Configuration

All tracking and display settings are managed through `set_config`. Settings are stored both in-memory and in `_config.conf` so they persist between sessions.

In [ ]:
# Show all available settings and their descriptions
print(AT.set_config.__doc__)

In [ ]:
# Print the current configuration
print(AT.config)

In [ ]:
# Typical setup for a new project
AT.set_config(
    show_tracking=False,
    userwait=False,
    frame_disstep=1,
    overwrite=True,
    create_vid=True,
    create_dat=True,
    link_dist=300,
    advanced=False,  # True enables additional B/W head/tail/skeleton calculations
)

**Shape filter settings** — useful when animals are small or change size during tracking:

In [ ]:
# size_filter_tol: accept areas between tol× and (1/tol)× the per-ID rolling median
# size_filter_memory: maximum accepted observations retained per ID
AT.set_config(size_filter=True, size_filter_tol=0.25, size_filter_memory=500)

#### 4. Set Number of Objects

In [ ]:
# Set how many objects to track per video (default is 1)
AT.set_objects(objects=[6, 2, 1, 4, 1])

In [ ]:
# Or set per-video using indices
AT.set_objects(inds=[0, 4], objects=[4, 1])

#### 5. Background Files

Background images are needed for background-subtraction (bw) tracking. ATracker samples random frames from the video to build a median background.

In [ ]:
AT.get_bgfiles(overwrite=False)

In [ ]:
# Only create backgrounds for a subset of frames (e.g. avoid first/last 500 frames)
AT.get_bgfiles(starts=[500], stops=[5000])

#### 6. Multiple Tracking Regions in One Video

When one video contains separate arenas that need independent tracking, use regions. Each region gets its own ROI and produces a separate output file.

In [ ]:
AT.set_config(regions=True)         # adds a 'region' column to the overview
AT.set_regions(inds=[3], nr=4)      # video at index 3 has 4 regions
AT.overview.loc[:, ["video", "region"]]

#### 7. Interactive Setup

The `set_interactive` function opens the visual editor for annotating tracking parameters. Use `inds`, `query`, or `cats` to target specific videos.

In [ ]:
# Pixel-to-mm conversion — draw a line of known length
AT.set_interactive(conv=True, conv_mm=(300,), inds=[2])
# Apply the conversion to all videos
AT.overview.conv = AT.overview.loc[2, "conv"]
AT.save()

In [ ]:
# Set start and stop frames for tracking
AT.set_interactive(framelimits=True, inds=[2])

In [ ]:
# Draw the region of interest
AT.set_interactive(roi=True)

In [ ]:
# Draw an exclusion mask (e.g. refuge area)
AT.set_interactive(mask=True, query='name=="solo"')

In [ ]:
# Draw arena walls (circular or irregular tanks)
AT.set_interactive(walls=True, query='name=="solo"')

In [ ]:
# Draw a secondary mask (e.g. structural elements)
AT.set_interactive(maskzone=True, query='name=="solo"')

In [ ]:
# Define multiple behavioural zones
AT.set_interactive(zones=True, query='name=="solo"')

In [ ]:
# Mark fixed points of interest (feeders, shelters, corners)
AT.set_interactive(getpts=True, ptcolnames=["c1", "c2", "c3"], query='name=="solo"')

#### 8. Thresholding Parameters

Calibrate detection parameters per tracking mode. Names beginning with "bw" (including "bwdark") use background subtraction; colour names such as "red" use HSV thresholding. Settings are stored in the project threshold YAML file.

The Editor exposes tracking gamma, blur, erosion, threshold, area and aspect-ratio limits. Explicit aspect-ratio values in a threshold configuration override AT.set_config() project defaults; tracking and the preview use the same resolved limits. See [Tuning automated tracking](3-tracking.md) for parameter guidance, flicker rejection and the rolling size filter.

In [ ]:
# BW background-subtraction thresholding
AT.set_interactive(threshtypes=["bw"], inds=[2])

In [ ]:
# Colour thresholding (one session per colour)
AT.set_interactive(threshtypes=["red", "blue", "green", "orange", "black", "brown"], inds=[0])

In [ ]:
# Alternative bw profile for a video with different lighting
AT.set_interactive(threshtypes=["bw2"], inds=[7])

In [ ]:
# Store a separate threshinfo file for a specific setup
AT.set_interactive(threshtypes=["bw"], inds=[7], threshfile="threshinfo2")

In [ ]:
# Assign threshold types per video in the overview and save
AT.overview.thresh_types = ["red,blue,green,orange,black,brown"] + ["bw"] * 4
AT.save()

#### 9. Tracking

##### 9.1 Dry run

Before committing to a full tracking run, use `drymode` to track short random clips and verify your settings are correct.

In [ ]:
AT.drymode(rand_filenr=7, rand_seqnr=3, rand_seqlen=100, suffix="dry", rerun=True)

##### 9.2 Run tracking

In [ ]:
# Track all videos in the todo folder
AT.track(folder="todo")

In [ ]:
# Track specific videos by index
AT.track(folder="originals", inds=[1, 2])

In [ ]:
# Track by video name
inds = AT.get_inds("animtest_pair_071024_S01_1150")
AT.track(folder="originals", inds=inds)

In [ ]:
# Override start/stop frames for this run only
AT.track(folder="originals", inds=[1, 2], frame_start=100, frame_stop=500)

In [ ]:
# Use a non-default threshtype or number of objects for this run
AT.track(folder="originals", inds=[7], threshtype="bw2", suffix="bw2")
AT.track(folder="originals", inds=[0], objects=2)

In [ ]:
# Use a custom threshinfo file
AT.track(folder="originals", threshfile="threshinfo2")

##### 9.3 Parallel tracking

Set `pools` to the number of CPU cores to use. Multiprocessing requires running from a terminal script, not from a Jupyter notebook.
Create a file `run_tracking.py` in your project folder:

In [ ]:
import atracker, os

def main():
    AT = atracker.ATracker(os.path.join(os.path.expanduser("~"), "Desktop", "tutorial"))
    AT.set_config(overwrite=False)
    AT.track(folder="originals", pools=4)

if __name__ == "__main__":
    main()

##### 9.4 Watch mode

Pass `watch=True` to keep tracking running continuously. After each pass the overview is reloaded from disk (picking up newly added rows), any untracked videos are processed, and then it sleeps before checking again. Stop with Ctrl+C.

In [ ]:
# In a terminal script:
AT.track(folder="originals", pools=4, watch=True, watch_interval=60)

#### 10. Data Checking

In [ ]:
# Interactively review and correct tracked data
AT.check_interactive(folder="tracked", names=["animtest_pair_071024_S01_1150"],
                     fileaction="overwrite")

#### 11. Data Processing

`process()` takes raw tracking CSVs and produces cleaned, enriched data: unit conversion, smoothing, trajectory gap-filling, distance calculations (to ROI, mask, walls, zones, points), and more.

In [ ]:
# Show all processing options
print(AT.process.__doc__)

In [ ]:
# Set the animal ID for a video in the overview before processing
AT.overview.loc[AT.get_inds("animtest_solo_071024_S01_1413"), "ID"] = 4
AT.save()

In [ ]:
AT.process(
    pools=1,
    names=["animtest_solo_071024_S01_1413"],
    overwrite=True,
    fulldata=True,
    convert=True,
    smoothwin=10,
    mask_margin=20,
    max_traj_gap=50,
    min_traj_len=10,
    roi_edge_margin=10,
    interp_gap_com=500,
    interp_gap_orient=100,
    orient_min_speed=1,
    interpolate=True,
    compute_movement=True,
    compute_distances=True,
)

#### 12. Manual Tracking

`manual_tracker` opens the visual editor for manual annotation or correction of trajectory data. It can load an existing CSV to continue editing, or start from scratch.

In [ ]:
from atracker import manual_tracker

manual_tracker(
    media_file="/path/to/video_TR.mp4",
    data_file="/path/to/video.csv",   # optional: load existing data
    mode="timepoints",
    firstframe=1,
    lastframe=None,
    fileaction="overwrite",
    width=1280,
    height=960,
)

#### 13. Batch Measurement

`batch_measure` lets you manually measure lengths (e.g. body size, distance) across a folder of images or videos using the visual editor. Useful for morphometrics and calibration.

In [ ]:
from atracker import batch_measure

sizedata = batch_measure(
    folder="/path/to/images",
    px_per_mm=None,               # or provide a known value to skip calibration
    ask_id=False,                 # False = use filenames as IDs
    out_csv="/path/to/output.csv",
)

In [ ]:
# Assign IDs manually after measuring
ids = [f"F{str(i).zfill(2)}" for i in range(1, len(sizedata) + 1)]
sizedata["ID"] = ids
sizedata